# Data cleaning

FAIR-MAST database cleaning, some pulses are flagged by more than one criteria.

In [1]:
import os
from datetime import date
import numpy as np
import pandas as pd
import requests

In [2]:
# Read parquet data
outputDir = 'MAST_info'

catalogCachePath = os.path.join(f'MAST_info/fairmast_shots_catalog.parquet')
catalog = pd.read_parquet(catalogCachePath).set_index('shot_id', drop=False)

print(f'{len(catalog)} shots pulled, {catalog["campaign"].nunique()} campaigns')
print(catalog['campaign'].value_counts())

15969 shots pulled, 5 campaigns
campaign
M7    5168
M6    3848
M5    2679
M8    2595
M9    1679
Name: count, dtype: int64


## Keep list of exclusion reasons per shot

In [3]:
exclusionReasons = {shotId: [] for shotId in catalog.index}

def flag(mask, reason):
    """
    Records reasons for shot exclusion when mask is True
    """
    mask = mask.reindex(catalog.index, fill_value=False)
    for shotId in catalog.index[mask]:
        exclusionReasons[shotId].append(reason)
    print(f'{int(mask.sum())} shots flagged: {reason}')

**Screening for:** shots with no data location in the catalog and duplicate shot IDs. 

In [4]:
noDataLocation = catalog['url'].isna() | catalog['endpoint_url'].isna()
flag(noDataLocation, 'no data location (url or endpoint_url missing)')

duplicateShotId = pd.Series(catalog.index.duplicated(keep='first'), index=catalog.index)
flag(duplicateShotId, 'duplicate shot_id in the catalog')

0 shots flagged: no data location (url or endpoint_url missing)
0 shots flagged: duplicate shot_id in the catalog


**Screening for:** RPM coils. TokaMaker's Grad-Shafranov solve is axisymmetric with no 3D coil model, so cant represent an RMP shot

From M8 onward `rmp_coil` is populated for most shots but blank for some. Shot log for some explicitly mentions adding or removing ELM coils mid campaign, so missing value cant be assumed off, treated as RMP on.

In [5]:
print(catalog['rmp_coil'].value_counts(dropna=False))
print(pd.crosstab(catalog['campaign'], catalog['rmp_coil'], dropna=False))

rmpFieldNotTracked = catalog['campaign'].isin(['M5', 'M6', 'M7'])
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('')).str.lower()
mentionsRmpCoils = comments.str.contains('elm coil|rmp coil', regex=True)
laterEraUnverified = catalog['rmp_coil'].isna() & ~rmpFieldNotTracked
print(f'{int((laterEraUnverified & mentionsRmpCoils).sum())} of {int(laterEraUnverified.sum())} unverified M8/M9 shots mention ELM/RMP coils in the logs')

rmpOnOrUnverified = (catalog['rmp_coil'] == True) | laterEraUnverified
flag(rmpOnOrUnverified, 'RMP coils on or unverified')

rmp_coil
None     12811
False     2639
True       519
Name: count, dtype: int64
rmp_coil  False  True   NaN
campaign                   
M5            0     0  2679
M6            0     0  3848
M7            0     0  5168
M8         1641   289   665
M9          998   230   451
101 of 1116 unverified M8/M9 shots mention ELM/RMP coils in the logs
1635 shots flagged: RMP coils on or unverified


**Screening for:** pellet injection, no pellet particle source in the pulse setup. no pellet timing or size data

In [6]:
print(catalog['pellets'].value_counts(dropna=False))
pelletsOnOrUnverified = catalog['pellets'] != False
flag(pelletsOnOrUnverified, 'pellets on or unverified')

pellets
False    15679
True       191
None        99
Name: count, dtype: int64
290 shots flagged: pellets on or unverified


**Screening for:** heating configuration the pulse setup has no source model for. Only Ohmic and the single or double neutral beam configurations supported.

`heating` unset for every shot in M5, M6 and M7. Two columns record when each beam started, `nbi_ss_start_time` and `nbi_sw_start_time` 

Verified these features against shots that already have heating label. Use same rule to determine heating label for unlabeled early shots. Anything still missing a label after this or not one of the four supported configurations is removed.

In [7]:
labeledLaterEra = catalog[catalog['campaign'].isin(['M8', 'M9']) & catalog['heating'].notna()]
validation = labeledLaterEra.groupby('heating')[['nbi_ss_start_time', 'nbi_sw_start_time']].apply(lambda g: g.notna().mean())
validation['n'] = labeledLaterEra.groupby('heating').size()
print('Fraction of each labeled heating category with an NBI start time set:\n')
print(validation, '\n')

def determineHeating(row):
    """
    Infers Ohmic/SS Beam/SW Beam/2 beams from which NBI start-time columns are set
    """
    hasSS = pd.notna(row['nbi_ss_start_time'])
    hasSW = pd.notna(row['nbi_sw_start_time'])
    if hasSS and hasSW:
        return '2 Beams,SS Beam,SW Beam'
    if hasSS:
        return 'SS Beam'
    if hasSW:
        return 'SW Beam'
    return 'Ohmic'

unlabeledEarlyEra = catalog['heating'].isna() & catalog['campaign'].isin(['M5', 'M6', 'M7'])
catalog['heatingResolved'] = catalog['heating']
catalog.loc[unlabeledEarlyEra, 'heatingResolved'] = catalog.loc[unlabeledEarlyEra].apply(determineHeating, axis=1)
print(f'{int(unlabeledEarlyEra.sum())} shots given an assumed heating label')
print(catalog['heatingResolved'].value_counts(dropna=False))

supportedHeating = {'Ohmic', 'SS Beam', 'SW Beam', '2 Beams,SS Beam,SW Beam'}
unsupportedHeating = ~catalog['heatingResolved'].isin(supportedHeating)
flag(unsupportedHeating, 'heating configuration missing or not supported')

Fraction of each labeled heating category with an NBI start time set:

                         nbi_ss_start_time  nbi_sw_start_time     n
heating                                                            
2 Beams,SS Beam,SW Beam           0.998282           0.998282  1164
Ohmic                             0.000000           0.000452  2210
SS Beam                           1.000000           0.007440   672
SW Beam                           0.000000           0.995455   220 

11695 shots given an assumed heating label
heatingResolved
Ohmic                      8582
2 Beams,SS Beam,SW Beam    2729
SW Beam                    2535
SS Beam                    2115
NaN                           8
Name: count, dtype: int64
8 shots flagged: heating configuration missing or not supported


**Screening for:** shots with no meaningful plasma current, shots that spike or disrupt early without normal ramp up, flat top, ramp down.

`plasma_time_avg_current` is the average current over the flat-top window. Ratio to peak current close to 1 for shot that held flattop. Compare this ratio for shots the operator log calls aborted, TF only, calibration, or no plasma.

In [8]:
badShotKeywords = ('no plasma', 'aborted', 'tf only', 'tf-only', 'glow discharge', 'calibration shot')
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('')).str.lower()
calledOutInLog = comments.apply(lambda text: any(keyword in text for keyword in badShotKeywords))
flatTopRatio = catalog['plasma_time_avg_current'] / catalog['plasma_max_current']
print('avg/peak current ratio, shots called out in the log as aborted, TF only, calibration or no plasma:')
print(flatTopRatio[calledOutInLog].describe())
print('same ratio, every other shot:')
print(flatTopRatio[~calledOutInLog].describe())

noMeaningfulCurrent = catalog['plasma_max_current'].isna() | (catalog['plasma_max_current'].abs() < 10)
flag(noMeaningfulCurrent, 'no recorded current or peak current under 10 kA')

didNotHoldFlatTop = flatTopRatio.isna() | (flatTopRatio < 0.7)
flag(didNotHoldFlatTop, 'current spiked or fell instead of holding a flat top (avg/peak ratio under 0.7)')

avg/peak current ratio, shots called out in the log as aborted, TF only, calibration or no plasma:
count    80.000000
mean      0.803210
std       0.539141
min      -0.983235
25%       0.476914
50%       0.834177
75%       1.100250
max       1.821595
dtype: float64
same ratio, every other shot:
count    13084.000000
mean         0.890168
std          0.383792
min         -1.047187
25%          0.952000
50%          0.978130
75%          0.986200
max          5.677265
dtype: float64
3645 shots flagged: no recorded current or peak current under 10 kA
4086 shots flagged: current spiked or fell instead of holding a flat top (avg/peak ratio under 0.7)


**Screening for:** MARFE events noted in the operator comments.

MARFE (Multifaceted Asymmetric Radiation From the Edge) is a localized edge radiation and cooling instability, not axisymmetric and not something TokaMaker's Grad-Shafranov solve or TORAX's 1D transport can represent. Same reasoning as the RMP and pellet screens above, no model for this physics in the pulse setup.

In [9]:
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('') + ' ' + catalog['shot_summary'].fillna('') + ' ' + catalog['shot_objective'].fillna('')).str.lower()
mentionsMarf = comments.str.contains('marf', regex=False)
negatedMarf = comments.str.contains('no marf|not marf|non-marf', regex=True)
print(f'{int(mentionsMarf.sum())} shots mention marf/marfe in the logs, {int(negatedMarf.sum())} phrased as not happening')

flag(mentionsMarf & ~negatedMarf, 'MARFE mentioned in operator comments, no edge radiation model')

44 shots mention marf/marfe in the logs, 1 phrased as not happening
43 shots flagged: MARFE mentioned in operator comments, no edge radiation model


**Screening for:** disruptions noted in the operator comments.

A disruption is a fast, non-axisymmetric loss of confinement, not something a quasi-static equilibrium plus 1D transport evolution can represent. Catches late disruptions near the end of an otherwise acceptable flat top that the avg/peak current ratio screen above can miss.

In [10]:
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('') + ' ' + catalog['shot_summary'].fillna('') + ' ' + catalog['shot_objective'].fillna('')).str.lower()
mentionsDisruption = comments.str.contains('disrupt', regex=False)
negatedDisruption = comments.str.contains('no disruption|non-disrupt|didn.?t disrupt|without disrupt', regex=True)
print(f'{int(mentionsDisruption.sum())} shots mention disruption in the logs, {int(negatedDisruption.sum())} phrased as not having one')

flag(mentionsDisruption & ~negatedDisruption, 'disruption mentioned in operator comments')

1345 shots mention disruption in the logs, 48 phrased as not having one
1297 shots flagged: disruption mentioned in operator comments


**Screening for:** pulses shorter than 0.45s.

A short pulse leaves little time for the equilibrium and profiles to evolve, not enough to compare against meaningfully.

In [11]:
tooShort = catalog['plasma_end_time'].isna() | (catalog['plasma_end_time'] < 0.45)
print(f'{int(tooShort.sum())} shots have less than 0.45s of plasma or a missing duration')

flag(tooShort, 'pulse shorter than 0.45s')

12012 shots have less than 0.45s of plasma or a missing duration
12012 shots flagged: pulse shorter than 0.45s


**Screening for:** MHD activity noted in the operator comments (locked modes, tearing modes, fishbones).

These are rotating or mode-locking instabilities, non-axisymmetric, not something TokaMaker's Grad-Shafranov solve or TORAX's 1D transport can represent.

In [12]:
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('') + ' ' + catalog['shot_summary'].fillna('') + ' ' + catalog['shot_objective'].fillna('')).str.lower()
mentionsMhd = comments.str.contains('locked mode|tearing|fishbone', regex=True)
negatedMhd = comments.str.contains('no locked mode|not locked|no tearing|no fishbone', regex=True)
print(f'{int(mentionsMhd.sum())} shots mention locked modes, tearing, or fishbones in the logs, {int(negatedMhd.sum())} phrased as not happening')

flag(mentionsMhd & ~negatedMhd, 'MHD activity (locked mode/tearing/fishbone) mentioned in operator comments')

747 shots mention locked modes, tearing, or fishbones in the logs, 9 phrased as not happening
738 shots flagged: MHD activity (locked mode/tearing/fishbone) mentioned in operator comments


**Screening for:** impurity contamination noted in the operator comments.

Pulse setup assumes a single fixed carbon impurity species and a frozen Zeff, a contamination event (oxygen influx, etc) isnt represented.

In [13]:
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('') + ' ' + catalog['shot_summary'].fillna('') + ' ' + catalog['shot_objective'].fillna('')).str.lower()
mentionsImpurity = comments.str.contains('oxygen|impurit', regex=True)
negatedImpurity = comments.str.contains('no oxygen|no impurit', regex=True)
print(f'{int(mentionsImpurity.sum())} shots mention oxygen or impurities in the logs, {int(negatedImpurity.sum())} phrased as not an issue')

flag(mentionsImpurity & ~negatedImpurity, 'impurity contamination mentioned in operator comments')

813 shots mention oxygen or impurities in the logs, 6 phrased as not an issue
807 shots flagged: impurity contamination mentioned in operator comments


**Screening for:** ECRH mentioned in operator comments, no heating source model and no power signal for it in the pulse setup.

In [ ]:
comments = (catalog['shot_preshot_comment'].fillna('') + ' ' + catalog['shot_postshot_comment'].fillna('') + ' ' + catalog['shot_summary'].fillna('') + ' ' + catalog['shot_objective'].fillna('')).str.lower()
mentionsEcrh = comments.str.contains('ecrh', regex=False)
negatedEcrh = comments.str.contains('ecrh postponed|no ecrh|ecrh cancelled|ecrh canceled', regex=True)
print(f'{int(mentionsEcrh.sum())} shots mention ecrh in the logs, {int(negatedEcrh.sum())} phrased as not happening')

flag(mentionsEcrh & ~negatedEcrh, 'ECRH mentioned in operator comments, no heating model or power signal for it')

## Aggregate reasons and mark usability

A shot is usable only if none of the screens above flagged it.

In [14]:
catalog['exclusionReasons'] = catalog.index.map(exclusionReasons)
catalog['usable'] = catalog['exclusionReasons'].apply(len) == 0
print(f'{int(catalog["usable"].sum())} of {len(catalog)} shots usable')
print(catalog.loc[~catalog['usable'], 'exclusionReasons'].explode().value_counts())

2194 of 15969 shots usable
exclusionReasons
pulse shorter than 0.45s                                                           12012
current spiked or fell instead of holding a flat top (avg/peak ratio under 0.7)     4086
no recorded current or peak current under 10 kA                                     3645
RMP coils on or unverified                                                          1635
disruption mentioned in operator comments                                           1297
impurity contamination mentioned in operator comments                                807
MHD activity (locked mode/tearing/fishbone) mentioned in operator comments           738
pellets on or unverified                                                             290
MARFE mentioned in operator comments, no edge radiation model                         43
heating configuration missing or not supported                                         8
Name: count, dtype: int64


## Write the cleaned and removed datasets

For removed pulses, keep shot with removal reason. Cleaned dataset keeps only the usable rows.

In [15]:
usableColumns = ['shot_id', 'campaign', 'heating', 'heatingResolved', 'shot_scenario', 'plasma_max_current', 'reference_shot']
screenedColumns = usableColumns + ['usable', 'exclusionReasons']

catalog['exclusionReasons'] = catalog['exclusionReasons'].apply(lambda reasons: '; '.join(reasons))

screenedPath = os.path.join(outputDir, 'screened_pulses.csv')
catalog[screenedColumns].to_csv(screenedPath, index=False)

cleanedPath = os.path.join(outputDir, 'usable_pulses.csv')
catalog.loc[catalog['usable'], usableColumns].to_csv(cleanedPath, index=False)

print(f'Screened pulses written to {screenedPath}')
print(f'Cleaned pulses written to {cleanedPath}')

Screened pulses written to MAST_info/screened_pulses.csv
Cleaned pulses written to MAST_info/usable_pulses.csv
